# Proposed hybrid LightGBM — revised evaluation

Feature construction follows the 4th-place domain-indicator ideas and the 61st-place
LightGBM temporal-feature pipeline. Sources: [61st-place notebook](https://www.kaggle.com/code/rafanikitas/hull-eda-training-pipeline),
`baselines/04_4th_writeup.md`; cross terms follow the Hull starter and 100th-place recipe.
The domain features are our reimplementation of an indicator family, not a disclosed alpha.

The executable pipeline lives in `src/experiments.py`; see `INTERFACE.md`.
The same analysis artifacts serve both model notebooks and `report_analysis.py`.

## Evaluation protocol

Four purged outer temporal folds estimate the performance of the entire selection
procedure. Model hyperparameters, blend choice and allocation rule are selected on
three inner temporal folds using only the outer training rows. Imputation statistics
are fitted per training fold. Tree counts are selected internally; outer validation
labels are never used for early stopping. All allocations are clipped to **[0, 2]**.
The proposed policy is fixed `naive_allocation(k=50)`; the improved policy is selected
internally from naive, binary, ternary, vol_target and passive w=1.

The public block was consulted during historical development (the early 0.011 score
motivated a change). It is a **diagnostic**, not a fresh independent test set.
The historical 0.453 → 0.770 comparison reused tuning folds and is not independent
evidence of improvement. Nested CV remains retrospective; no new future data is available.

Run `python report_analysis.py` from the root first to generate results. A fresh
notebook run generates them if absent. Default budgets are explicit in the manifest;
these revised experiments do not reuse the original 120/40/25/30-trial searches.

Rolling blend weights use a 60-row causal warm-up from predictions of the same fitted models.
Warm-up rows are excluded from validation objectives and evaluation, with separate history per fold.


In [ ]:
import sys
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/experiments.py").exists())
sys.path.insert(0, str(root))
import json
import hashlib
import pandas as pd
from IPython.display import display, Image
from data_checks import run_checks
from report_analysis import build_report
from src import RESULTS_DIR, compare
from src.experiments import ExperimentConfig, run_experiment

directory = RESULTS_DIR / "analysis"
checks = run_checks(output_dir=directory)
if not (directory / "manifest.json").exists():
    run_experiment(directory, config=ExperimentConfig())
    build_report(directory)
manifest = json.loads((directory / "manifest.json").read_text())
assert manifest["data_sha256"] == checks["data_sha256"], "Dataset changed; rerun report_analysis.py"
assert manifest.get("pipeline_sha256") == hashlib.sha256((root / "src/experiments.py").read_bytes()).hexdigest(), "Pipeline changed; rerun report_analysis.py"
assert manifest["config"]["seed_count"] >= 20, "Smoke results are not report results"
metrics = pd.read_csv(directory / "fold_metrics.csv")


## Selection provenance

Every selection range ends strictly before its outer evaluation range.


In [ ]:
selections = pd.DataFrame(manifest["outer_selections"])
display(selections[["outer_fold", "selection_max_date_id", "evaluation_min_date_id", "blend", "allocation"]])
print(manifest["config"])


## Nested outer results, including passive w=1


In [ ]:
display(metrics.loc[(metrics.split == "cv") & metrics.model.isin(["proposed", "passive"])][["fold", "model", "modified_sharpe", "spearman_ic", "vol_ratio"]])
display(compare(split="cv"))


## Domain-feature ablation over 20 fit seeds

Within each outer fold, the hyperparameters selected on its inner folds are held
fixed. Seeds 0–19 are paired with and without the domain block. This measures fit-seed
sensitivity conditional on those parameters; it does not repeat the entire search
20 times or quantify uncertainty over new market periods. Importance uses actual
LightGBM **gain**, not split counts, and does not establish predictive benefit.


In [ ]:
display(pd.read_csv(directory / "ablation_seeds.csv"))
print(json.loads((directory / "ablation_summary.json").read_text()))
display(pd.read_csv(directory / "feature_gain_blocks.csv"))


## Public diagnostic

The index row is included in every comparison. Baselines have public-only evaluations
and keep their original preprocessing and internal splitters; no baseline CV rows are implied.


In [ ]:
display(compare(split="public"))
display(Image(filename=str(directory / "figures/figure_4a.png")))
display(Image(filename=str(directory / "figures/figure_4b.png")))
display(Image(filename=str(directory / "figures/figure_5.png")))
